In [1]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 4.7 MB/s eta 0:00:00


In [4]:
import pandas as pd

# Veriyi oku
df = pd.read_csv('train.csv')
print(df.head())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

Bu veri setinde MSZoning, Street, Alley, LandContour gibi sütunlar vardır. Bunlar kategorik verilerdir. Örneğin MSZoning sütunundaki değerler (RL, RM, FV...) arasında bir "sıralama" veya "büyüklük" ilişkisi yoktur. RL'nin RM'den daha "büyük" bir sayı olduğunu söyleyemeyiz; bunlar sadece farklı bölgelerdir.

İşte bu yüzden, bilgisayarın bu farklılığı "sayısal" ama "eşit" bir şekilde anlaması için One-Hot Encoding yaparız.

In [5]:
print(df['MSZoning'].value_counts())

MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64


In [6]:
from feature_engine.encoding import RareLabelEncoder

In [11]:
# Verinin %5'inden az olanları 'Rare' yapalım
rare_encoder = RareLabelEncoder(tol=0.05, n_categories=2, variables=['MSZoning'])
# Uygula
df = rare_encoder.fit_transform(df)

Burada ne kadar MSZoning olduğunu anlamaya çalışıyorum ve ardındadanda onehotencoder yaparak hepsini 0 1 formatına çevircem

In [12]:
# Şimdi tekrar bak bakalım ne değişmiş?
print(df['MSZoning'].value_counts())

MSZoning
RL      1151
RM       218
Rare      91
Name: count, dtype: int64


In [13]:
from feature_engine.encoding import OneHotEncoder

# One-Hot işlemini başlatalım
ohe = OneHotEncoder(variables=['MSZoning'])

In [14]:
# Dönüştür
df_final = ohe.fit_transform(df)

In [15]:
# Sonucu kontrol et (Sütunlar 0 ve 1 oldu mu?)
print(df_final.head())

   Id  MSSubClass  LotFrontage  LotArea Street Alley LotShape LandContour  \
0   1          60         65.0     8450   Pave   NaN      Reg         Lvl   
1   2          20         80.0     9600   Pave   NaN      Reg         Lvl   
2   3          60         68.0    11250   Pave   NaN      IR1         Lvl   
3   4          70         60.0     9550   Pave   NaN      IR1         Lvl   
4   5          60         84.0    14260   Pave   NaN      IR1         Lvl   

  Utilities LotConfig  ... MiscFeature MiscVal MoSold YrSold SaleType  \
0    AllPub    Inside  ...         NaN       0      2   2008       WD   
1    AllPub       FR2  ...         NaN       0      5   2007       WD   
2    AllPub    Inside  ...         NaN       0      9   2008       WD   
3    AllPub    Corner  ...         NaN       0      2   2006       WD   
4    AllPub       FR2  ...         NaN       0     12   2008       WD   

  SaleCondition  SalePrice  MSZoning_RL  MSZoning_RM  MSZoning_Rare  
0        Normal     208500  

In [19]:
from feature_engine.encoding import OrdinalEncoder

# 1. Önce bu sütunda hangi değerler var bakalım
print("Dönüşüm öncesi değerler:")
print(df['ExterQual'].unique())

# 2. Ordinal Encoder'ı kuralım
# encoding_method='arbitrary' dersek 0, 1, 2 diye rastgele verir.
# Biz 'ordered' diyerek hedef değişkene (fiyata) göre sıralatabiliriz
# ya da sözlükle kendimiz puanlayabiliriz. Şimdilik kursa uygun 'arbitrary' gidelim:
ordinal_enc = OrdinalEncoder(encoding_method='arbitrary', variables=['ExterQual'])

# 3. Uygulayalım
df = ordinal_enc.fit_transform(df)

Dönüşüm öncesi değerler:
['Gd' 'TA' 'Ex' 'Fa']

Dönüşüm sonrası değerler:


In [20]:
print(df['ExterQual'].head())

0    0
1    1
2    0
3    1
4    0
Name: ExterQual, dtype: int64


ExterQual sütununda şu kelimeler vardı:

Ex (Mükemmel)

Gd (İyi)

TA (Ortalama)

Fa (İdare Eder)

Bilgisayar bu kelimeleri toplama-çıkarmaya sokamaz. Eğer biz bunları One-Hot Encoding yapsaydık (önceki yaptığımız gibi), bilgisayar bunların arasındaki kalite farkını anlayamazdı. Sadece "farklılar" derdi.

2. Çözümümüz: Ordinal Encoding (Sıralama)
Ordinal Encoding'de biz bilgisayara şunu deriz: "Bak, bu kelimeler arasında bir hiyerarşi var. En iyisi en büyük rakamı alsın."

Sende çıkan [0 1 2 3] sayıları şu anlama geliyor:

3: En yüksek kaliteli evler (Muhtemelen Ex)

2: Bir tık altı (Gd)

1: Ortalama (TA)

0: En düşük kaliteli olanlar (Fa)

In [24]:
# Sütundaki benzersiz değerleri görelim
print("Kalite değerleri:", df['ExterQual'].unique())

Kalite değerleri: [0 1 2 3]


1. Mantık: "Popülerlik Sayıları"
Bilgisayara mahallenin adını vermek yerine, o mahalleden veri setinde kaç tane ev olduğunu söylüyoruz.

North Ames: 225 kez geçiyorsa, mahalle ismini silip yerine 225 yazıyoruz.

Veenker: 11 kez geçiyorsa, yerine 11 yazıyoruz.

Böylece model, "Hangi mahallede daha çok satış yapılmış (arz-talep dengesi)" bilgisini sayısal olarak öğrenir.

In [25]:
from feature_engine.encoding import CountFrequencyEncoder

# 1. Encoder'ı kuralım
# encoding_method='count' -> Mahallenin toplam sayısını yazar
# encoding_method='frequency' -> Mahallenin tüm veri içindeki % oranını yazar
count_enc = CountFrequencyEncoder(
    encoding_method='count',
    variables=['Neighborhood']
)

In [26]:
# 2. Eğit ve Uygula
df = count_enc.fit_transform(df)

In [27]:
# 3. Sonucu Kontrol Et
print("Neighborhood sütunu (Count Encoding sonrası):")
print(df['Neighborhood'].head())

Neighborhood sütunu (Count Encoding sonrası):
0    150
1     11
2    150
3     51
4     41
Name: Neighborhood, dtype: int64


In [28]:
# Hangi mahallenin kaç puan aldığını görmek istersen:
print("\nKodlama Sözlüğü:")
print(count_enc.encoder_dict_)


Kodlama Sözlüğü:
{'Neighborhood': {'NAmes': 225, 'CollgCr': 150, 'OldTown': 113, 'Edwards': 100, 'Somerst': 86, 'Gilbert': 79, 'NridgHt': 77, 'Sawyer': 74, 'NWAmes': 73, 'SawyerW': 59, 'BrkSide': 58, 'Crawfor': 51, 'Mitchel': 49, 'NoRidge': 41, 'Timber': 38, 'IDOTRR': 37, 'ClearCr': 28, 'SWISU': 25, 'StoneBr': 25, 'Blmngtn': 17, 'MeadowV': 17, 'BrDale': 16, 'Veenker': 11, 'NPkVill': 9, 'Blueste': 2}}


1. Mantık: Sayıların Diliyle Konuşmak
Diyelim ki İstanbul'da ev fiyatlarını tahmin ediyorsun ve elinde "Semt" sütunu var:

Beşiktaş: Bu semtteki evlerin ortalama fiyatı 10.000.000 TL.

Esenyurt: Bu semtteki evlerin ortalama fiyatı 2.000.000 TL.

Target Encoding şunu yapar: Veri setindeki tüm "Beşiktaş" yazılarını siler ve yerine 10000000 yazar. Tüm "Esenyurt" yazılarını siler ve yerine 2000000 yazar.

2. Neden Bu Kadar Güçlü?
Boyut Koruma: One-Hot Encoding gibi 500 tane mahalle için 500 tane sütun açmaz. Tek bir sütunda kalır ama o sütuna devasa bir bilgi yükler.

Doğrudan İlişki: Modelin "Hangi mahalle daha pahalı?" diye keşfetmesine gerek kalmaz; biz ona zaten mahallenin fiyat gücünü sayısal olarak vermiş oluruz.

In [35]:
# 1. Veriyi en baştan (temiz haliyle) tekrar oku
df = pd.read_csv('train.csv')

In [36]:
from feature_engine.encoding import MeanEncoder

# 2. Sadece Neighborhood üzerinde Target Encoding yapalım
target_enc = MeanEncoder(variables=['Neighborhood'])

# 3. ŞİMDİ ÇALIŞACAK: Çünkü Neighborhood şu an tekrar 'metin' (object) formatında
target_enc.fit(df, df['SalePrice'])

# 4. Dönüştür
df_target = target_enc.transform(df)

# Kontrol et: Mahalleler fiyat ortalaması oldu mu?
print(df_target['Neighborhood'].head())

0    197965.773333
1    238772.727273
2    197965.773333
3    210624.725490
4    335295.317073
Name: Neighborhood, dtype: float64


1. Neden İhtiyaç Duyarız? (Sorun)
House Prices veri setindeki MSZoning örneğini hatırla:

RL: 1151 ev

RM: 218 ev

C (all): Sadece 10 ev
Eğer sen modeline sadece 10 tane örneği olan C (all) kategorisini ayrı bir sınıf olarak verirsen, model bu 10 eve bakarak çok keskin ve yanlış kararlar verebilir (Overfitting). Ayrıca One-Hot Encoding yaparken sadece 10 ev için koca bir sütun açmak tablonu gereksiz şişirir.

2. Nasıl Çalışır? (Çözüm)
Bir eşik değer (threshold) belirlersin (Örneğin: %5). Verinin genelinde %5'ten daha az payı olan tüm kategorileri bir çuvala doldurup ağzını bağlarsın.

Öncesi:

İstanbul (%40)

Ankara (%30)

İzmir (%25)

Elazığ (%2) -> Nadir

Bolu (%2) -> Nadir

Uşak (%1) -> Nadir
Rare Label Sonrası:

İstanbul (%40)

Ankara (%30)

İzmir (%25)

Rare (%5) (Elazığ, Bolu ve Uşak birleşti)

In [38]:
from feature_engine.encoding import RareLabelEncoder

# 1. Ayarları yapalım
# tol=0.05 -> %5'ten az olanları nadir say
# n_categories=4 -> Sadece 4'ten fazla farklı kategori varsa bu işlemi yap
rare_enc = RareLabelEncoder(tol=0.05, n_categories=4, variables=['Neighborhood'])

# 2. Uygula
df = rare_enc.fit_transform(df)